# Desafio

A empresa Você Mais Seguro fornece planos de saúde e seguros de vida. 
A empresa está interessada em lança um novo seguro de veiculos, e desja entender a propensão de compra de seus clientes para este novo cenário. 
Em uma primeira etapa, foi realizada uma grande pesquisa com seus cleintes, para entender quais destes clientes estariam interessados em adquirir o novo seguro de veiculos.

Agora, com novos clientes do plano de saude entrando para sua base, a empresa deseja entender quais destes novos clientes teriam maior propensão de compra do novo seguro de veiculos.
No entanto, a empresa não esta interessada em gastar com pesquisas para todos os novos clientes, e sim apenas para aqueles que tem maior propensão de compra, 
haja visto,que seu efetivo comercial, não é capaz de entrar em contato, para fornecer um processo de pré vendas personalizado para toda a base de seus novos clientes, somente
para 2000 deles de cada vez. 

Para isso, este projeto tem como objetivo apresentar uma lista, com os 2000 clientes com maior probabilidade de venda do seguro de veículos.

# 0.0 Imports

In [1]:
import pandas as pd 
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier 


### 0.1 Helper Functions

In [2]:
def jupyter_settings():
    %matplotlib inline
    %pylab inline

    plt.style.use( 'bmh' )
    plt.rcParams['figure.figsize'] = [25, 8]
    plt.rcParams['font.size'] = 24
    pd.options.display.max_columns = None
    pd.options.display.max_rows = None
    pd.set_option( 'display.expand_frame_repr', False )
    sns.set()

jupyter_settings()

def print_descriptive_stats(df,
                            column):
    print(f'**Clientes que adquiririam o Seguro - {column} statistics**')
    print(df[df.Response == 1][column].describe())
    print('-'*45)
    print(f'**Clientes NAO que adquiririam o Seguro - {column} statistics**')
    print(df[df.Response == 0][column].describe())
    print('\n')

def get_frequency_table(df, column): 

    df_frequency_1 = df[df.Response == 1].groupby(column).Response.count().reset_index()
    df_frequency_1.rename(columns={'Response':'Count_Response_1'}, inplace=True)


    df_frequency_0 = df[df.Response == 0].groupby(column).Response.count().reset_index()
    df_frequency_0.rename(columns={'Response':'Count_Response_0'}, inplace=True)

    df_frequency = pd.merge(df_frequency_1, df_frequency_0, on=column, how='inner') 
    df_frequency['Total'] = df_frequency['Count_Response_1'] + df_frequency['Count_Response_0']
    df_frequency['Propensity_to_buy'] = df_frequency['Count_Response_1']/df_frequency['Total']
    df_frequency['%_sample_to_buy'] = df_frequency['Count_Response_1']/df_frequency['Count_Response_1'].sum() 
    df_frequency['%_sample_to_not_buy'] = df_frequency['Count_Response_0']/df_frequency['Count_Response_0'].sum() 
    df_frequency['%_sample_size']   = df_frequency['Total']/len(df)
    df_frequency['%_general_propensity_to_buy'] = len(df[df.Response==1])/len(df)
    
    return df_frequency


def transform_pipeline(df):

    '''
    [EM FASE DE VALIDACAO....]
    '''
    from sklearn.preprocessing import StandardScaler, MinMaxScaler 


    region_code = df.Region_Code.copy()
    dummies_region_code = pd.get_dummies(region_code,prefix = "region_code")

    sales_channel = df.Policy_Sales_Channel.copy()
    dummies_sales_channel = pd.get_dummies(sales_channel,prefix = "policy_sales_channel")

    df_copy = df.copy()
    df_encoded = pd.concat([df_copy, dummies_sales_channel, dummies_region_code], axis = 1)


    df_encoded.drop(columns = ['Region_Code','Policy_Sales_Channel'], inplace = True)

    map_sexo_feminino = {
        'Male' : 0,
        'Female' : 1 
    }

    df_encoded['sexo_feminino'] = df_encoded.Gender.map(map_sexo_feminino)


    df_encoded.Vehicle_Damage = df_encoded.Vehicle_Damage.map({
        'Yes' : 1,
        'No'  : 0
    })


    scaller_premium      = MinMaxScaler()
    scaller_vintage      = MinMaxScaler()
    scaller_age          = MinMaxScaler()
    scaller_faixa_etaria = MinMaxScaler()

    df_encoded.Vintage_Scalled        = scaller_vintage.fit_transform(df_encoded.Vintage.values.reshape(-1,1))
    df_encoded.Annual_Premium_Scalled = scaller_premium.fit_transform(df_encoded.Annual_Premium.values.reshape(-1,1))


    df_encoded.Age        = scaller_age.fit_transform(df_encoded.Age.values.reshape(-1,1))
    df_encoded.faixa_etaria        = scaller_faixa_etaria.fit_transform(df_encoded.faixa_etaria.values.reshape(-1,1))


    df_encoded.Annual_Premium.describe()

    df_to_model = df_encoded[[x for x in df_encoded.columns if x not in ['id','Vintage_months','Vintage_weeks','Vintage']]]
    df_to_model.columns
    return df_to_model

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


# 1.0 Limpeza dos Dados

**Tipos de limpezas**

* Nulos
* Tipagem inconsistente 
* Claros outliers (indicio de erro de digitação)


### 1.1 - Conhecendo o Dataset

In [3]:
df_raw = pd.read_csv('../data/raw/data.csv')
df_raw.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


# 3.0 - Preparação dos dados

In [4]:
df_3 = df_raw.copy()
df_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  object 
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  object 
 7   Vehicle_Damage        381109 non-null  object 
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 34.9+ MB


### 3.1 - Criação de features

* Faixa etaria
* Tem entre 30 - 50 anos (EDA)

In [5]:
df_3['faixa_etaria'] = df_3.Age.apply(lambda x: 10*(x//10))
df_3['entre_30_50'] = df_3.Age.apply(lambda x: 1 if (x>=30) and (x<=50) else 0)


### 3.1 - Encodings

#### 3.1.1 - One Hot Encoding (dummi)

* Policy Sales Channel 
* Region Code

**Region Code**

A justificativa se da pela ausencia de elação quantitativa nos dados de region code. 

A regiao "102" ser maior que a regiao "32" não faz sentido


In [6]:
region_code = df_3.Region_Code.copy()
dummies_region_code = pd.get_dummies(region_code,prefix = "region_code")
dummies_region_code.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 53 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   region_code_0.0   381109 non-null  bool 
 1   region_code_1.0   381109 non-null  bool 
 2   region_code_2.0   381109 non-null  bool 
 3   region_code_3.0   381109 non-null  bool 
 4   region_code_4.0   381109 non-null  bool 
 5   region_code_5.0   381109 non-null  bool 
 6   region_code_6.0   381109 non-null  bool 
 7   region_code_7.0   381109 non-null  bool 
 8   region_code_8.0   381109 non-null  bool 
 9   region_code_9.0   381109 non-null  bool 
 10  region_code_10.0  381109 non-null  bool 
 11  region_code_11.0  381109 non-null  bool 
 12  region_code_12.0  381109 non-null  bool 
 13  region_code_13.0  381109 non-null  bool 
 14  region_code_14.0  381109 non-null  bool 
 15  region_code_15.0  381109 non-null  bool 
 16  region_code_16.0  381109 non-null  bool 
 17  region_cod

**Policy Sales Channel**

In [7]:

sales_channel = df_3.Policy_Sales_Channel.copy()
dummies_sales_channel = pd.get_dummies(sales_channel,prefix = "policy_sales_channel")
dummies_sales_channel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Columns: 155 entries, policy_sales_channel_1.0 to policy_sales_channel_163.0
dtypes: bool(155)
memory usage: 56.3 MB


In [8]:
dummies_sales_channel.head(5)

,policy_sales_channel_1.0,policy_sales_channel_2.0,policy_sales_channel_3.0,policy_sales_channel_4.0,policy_sales_channel_6.0,policy_sales_channel_7.0,policy_sales_channel_8.0,policy_sales_channel_9.0,policy_sales_channel_10.0,policy_sales_channel_11.0,policy_sales_channel_12.0,policy_sales_channel_13.0,policy_sales_channel_14.0,policy_sales_channel_15.0,policy_sales_channel_16.0,policy_sales_channel_17.0,policy_sales_channel_18.0,policy_sales_channel_19.0,policy_sales_channel_20.0,policy_sales_channel_21.0,policy_sales_channel_22.0,policy_sales_channel_23.0,policy_sales_channel_24.0,policy_sales_channel_25.0,policy_sales_channel_26.0,policy_sales_channel_27.0,policy_sales_channel_28.0,policy_sales_channel_29.0,policy_sales_channel_30.0,policy_sales_channel_31.0,policy_sales_channel_32.0,policy_sales_channel_33.0,policy_sales_channel_34.0,policy_sales_channel_35.0,policy_sales_channel_36.0,policy_sales_channel_37.0,policy_sales_channel_38.0,policy_sales_channel_39.0,policy_sales_channel_40.0,policy_sales_channel_41.0,policy_sales_channel_42.0,policy_sales_channel_43.0,policy_sales_channel_44.0,policy_sales_channel_45.0,policy_sales_channel_46.0,policy_sales_channel_47.0,policy_sales_channel_48.0,policy_sales_channel_49.0,policy_sales_channel_50.0,policy_sales_channel_51.0,policy_sales_channel_52.0,policy_sales_channel_53.0,policy_sales_channel_54.0,policy_sales_channel_55.0,policy_sales_channel_56.0,policy_sales_channel_57.0,policy_sales_channel_58.0,policy_sales_channel_59.0,policy_sales_channel_60.0,policy_sales_channel_61.0,policy_sales_channel_62.0,policy_sales_channel_63.0,policy_sales_channel_64.0,policy_sales_channel_65.0,policy_sales_channel_66.0,policy_sales_channel_67.0,policy_sales_channel_68.0,policy_sales_channel_69.0,policy_sales_channel_70.0,policy_sales_channel_71.0,policy_sales_channel_73.0,policy_sales_channel_74.0,policy_sales_channel_75.0,policy_sales_channel_76.0,policy_sales_channel_78.0,policy_sales_channel_79.0,policy_sales_channel_80.0,policy_sales_channel_81.0,policy_sales_channel_82.0,policy_sales_channel_83.0,policy_sales_channel_84.0,policy_sales_channel_86.0,policy_sales_channel_87.0,policy_sales_channel_88.0,policy_sales_channel_89.0,policy_sales_channel_90.0,policy_sales_channel_91.0,policy_sales_channel_92.0,policy_sales_channel_93.0,policy_sales_channel_94.0,policy_sales_channel_95.0,policy_sales_channel_96.0,policy_sales_channel_97.0,policy_sales_channel_98.0,policy_sales_channel_99.0,policy_sales_channel_100.0,policy_sales_channel_101.0,policy_sales_channel_102.0,policy_sales_channel_103.0,policy_sales_channel_104.0,policy_sales_channel_105.0,policy_sales_channel_106.0,policy_sales_channel_107.0,policy_sales_channel_108.0,policy_sales_channel_109.0,policy_sales_channel_110.0,policy_sales_channel_111.0,policy_sales_channel_112.0,policy_sales_channel_113.0,policy_sales_channel_114.0,policy_sales_channel_115.0,policy_sales_channel_116.0,policy_sales_channel_117.0,policy_sales_channel_118.0,policy_sales_channel_119.0,policy_sales_channel_120.0,policy_sales_channel_121.0,policy_sales_channel_122.0,policy_sales_channel_123.0,policy_sales_channel_124.0,policy_sales_channel_125.0,policy_sales_channel_126.0,policy_sales_channel_127.0,policy_sales_channel_128.0,policy_sales_channel_129.0,policy_sales_channel_130.0,policy_sales_channel_131.0,policy_sales_channel_132.0,policy_sales_channel_133.0,policy_sales_channel_134.0,policy_sales_channel_135.0,policy_sales_channel_136.0,policy_sales_channel_137.0,policy_sales_channel_138.0,policy_sales_channel_139.0,policy_sales_channel_140.0,policy_sales_channel_143.0,policy_sales_channel_144.0,policy_sales_channel_145.0,policy_sales_channel_146.0,policy_sales_channel_147.0,policy_sales_channel_148.0,policy_sales_channel_149.0,policy_sales_channel_150.0,policy_sales_channel_151.0,policy_sales_channel_152.0,policy_sales_channel_153.0,policy_sales_channel_154.0,policy_sales_channel_155.0,policy_sales_channel_156.0,policy_sales_channel_157.0,policy_sales_channel_158.

**Merge ao dataset original**

In [9]:
df_3_copy = df_3.copy()
df_3_encoded = pd.concat([df_3_copy, dummies_sales_channel, dummies_region_code], axis = 1)


df_3_encoded.drop(columns = ['Region_Code','Policy_Sales_Channel'], inplace = True)
df_3_encoded.shape

(381109, 220)

**Region Code**

#### 3.1.2 - Binary Encoding

* Gender
* Driving License
* Previously Insured
* Vehicle Damage
* Response

In [10]:
df_3_encoded.Gender.value_counts()

Gender
Male      206089
Female    175020
Name: count, dtype: int64

In [11]:

map_sexo_feminino = {
    'Male' : 0,
    'Female' : 1 
}

df_3_encoded['sexo_feminino'] = df_3_encoded.Gender.map(map_sexo_feminino)


**Checagem dos demais campos**

In [12]:
df_3_encoded.Driving_License.value_counts()

Driving_License
1    380297
0       812
Name: count, dtype: int64

In [13]:
df_3_encoded.Previously_Insured.value_counts()

Previously_Insured
0    206481
1    174628
Name: count, dtype: int64

In [14]:
df_3_encoded.Vehicle_Damage.value_counts()

Vehicle_Damage
Yes    192413
No     188696
Name: count, dtype: int64

In [15]:
df_3_encoded.Vehicle_Damage = df_3_encoded.Vehicle_Damage.map({
    'Yes' : 1,
    'No'  : 0
})

df_3_encoded.Vehicle_Damage.value_counts()

Vehicle_Damage
1    192413
0    188696
Name: count, dtype: int64

In [16]:
df_3_encoded.Response.value_counts()

Response
0    334399
1     46710
Name: count, dtype: int64

### 3.1.3 - Ordinal Encoding

In [17]:
df_3_encoded.loc[:,'Vehicle_Age'] = df_3_encoded.loc[:,'Vehicle_Age'].map({'< 1 Year':0, '1-2 Year':1, '> 2 Years':2})


## 3.2 - Variaveis Numéricas (Padronização,  Normalização, transformação ciclica, etc.)

In [18]:
df_3_encoded[[x for x in df_3_encoded.columns if '.0' not in x]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  381109 non-null  int64  
 1   Gender              381109 non-null  object 
 2   Age                 381109 non-null  int64  
 3   Driving_License     381109 non-null  int64  
 4   Previously_Insured  381109 non-null  int64  
 5   Vehicle_Age         381109 non-null  object 
 6   Vehicle_Damage      381109 non-null  int64  
 7   Annual_Premium      381109 non-null  float64
 8   Vintage             381109 non-null  int64  
 9   Response            381109 non-null  int64  
 10  faixa_etaria        381109 non-null  int64  
 11  entre_30_50         381109 non-null  int64  
 12  sexo_feminino       381109 non-null  int64  
dtypes: float64(1), int64(10), object(2)
memory usage: 37.8+ MB


In [19]:
df_3_encoded.Vintage.values.shape

(381109,)

In [20]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler 


scaller_premium      = MinMaxScaler()
scaller_vintage      = MinMaxScaler()
scaller_age          = MinMaxScaler()
scaller_faixa_etaria = MinMaxScaler()

df_3_encoded.Vintage_Scalled        = scaller_vintage.fit_transform(df_3_encoded.Vintage.values.reshape(-1,1))
df_3_encoded.Annual_Premium_Scalled = scaller_premium.fit_transform(df_3_encoded.Annual_Premium.values.reshape(-1,1))


df_3_encoded.Age        = scaller_age.fit_transform(df_3_encoded.Age.values.reshape(-1,1))
df_3_encoded.faixa_etaria        = scaller_faixa_etaria.fit_transform(df_3_encoded.faixa_etaria.values.reshape(-1,1))


df_3_encoded.Annual_Premium.describe()

/tmp/ipykernel_7346/850787241.py:9: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_3_encoded.Vintage_Scalled        = scaller_vintage.fit_transform(df_3_encoded.Vintage.values.reshape(-1,1))
/tmp/ipykernel_7346/850787241.py:10: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_3_encoded.Annual_Premium_Scalled = scaller_premium.fit_transform(df_3_encoded.Annual_Premium.values.reshape(-1,1))


count    381109.000000
mean      30564.389581
std       17213.155057
min        2630.000000
25%       24405.000000
50%       31669.000000
75%       39400.000000
max      540165.000000
Name: Annual_Premium, dtype: float64

In [21]:
df_to_model = df_3_encoded[[x for x in df_3_encoded.columns if x not in ['id','Vintage_months','Vintage_weeks','Vintage']]]
df_to_model.columns


Index(['Gender', 'Age', 'Driving_License', 'Previously_Insured', 'Vehicle_Age',
       'Vehicle_Damage', 'Annual_Premium', 'Response', 'faixa_etaria',
       'entre_30_50',
       ...
       'region_code_44.0', 'region_code_45.0', 'region_code_46.0',
       'region_code_47.0', 'region_code_48.0', 'region_code_49.0',
       'region_code_50.0', 'region_code_51.0', 'region_code_52.0',
       'sexo_feminino'],
      dtype='object', length=219)

### 3.3 - Seleção de variaveis

In [22]:
def transform_pipeline(df):

    '''
    [EM FASE DE VALIDACAO....]
    '''
    from sklearn.preprocessing import StandardScaler, MinMaxScaler 


    region_code = df.Region_Code.copy()
    dummies_region_code = pd.get_dummies(region_code,prefix = "region_code")

    sales_channel = df.Policy_Sales_Channel.copy()
    dummies_sales_channel = pd.get_dummies(sales_channel,prefix = "policy_sales_channel")

    df_copy = df.copy()
    df_encoded = pd.concat([df_copy, dummies_sales_channel, dummies_region_code], axis = 1)

    df_encoded.loc[:,'Vehicle_Age'] = df_encoded.loc[:,'Vehicle_Age'].map({'< 1 Year':0, '1-2 Year':1, '> 2 Years':2})


    df_encoded.drop(columns = ['Region_Code','Policy_Sales_Channel'], inplace = True)

    map_sexo_feminino = {
        'Male' : 0,
        'Female' : 1 
    }

    df_encoded['sexo_feminino'] = df_encoded.Gender.map(map_sexo_feminino)


    df_encoded.Vehicle_Damage = df_encoded.Vehicle_Damage.map({
        'Yes' : 1,
        'No'  : 0
    })


    scaller_premium      = MinMaxScaler()
    scaller_vintage      = MinMaxScaler()
    scaller_age          = MinMaxScaler()
    scaller_faixa_etaria = MinMaxScaler()

    df_encoded.Vintage_Scalled        = scaller_vintage.fit_transform(df_encoded.Vintage.values.reshape(-1,1))
    df_encoded.Annual_Premium_Scalled = scaller_premium.fit_transform(df_encoded.Annual_Premium.values.reshape(-1,1))


    df_encoded.Age        = scaller_age.fit_transform(df_encoded.Age.values.reshape(-1,1))
    df_encoded.faixa_etaria        = scaller_faixa_etaria.fit_transform(df_encoded.faixa_etaria.values.reshape(-1,1))


    df_encoded.Annual_Premium.describe()

    df_to_model = df_encoded[[x for x in df_encoded.columns if x not in ['id','Vintage_months','Vintage_weeks','Vintage']]]

    return df_to_model

#df_to_model = transform_pipeline(df_raw)

In [23]:
from sklearn.tree import DecisionTreeClassifier
df_to_model = df_3_encoded.copy()
df_to_model_train, df_to_model_test = train_test_split(df_to_model, test_size=0.3, 
                                                       random_state=42, stratify=df_to_model['Response'])

inference_cols = [x for x in df_to_model.columns if x not in ('Response','Gender')]
model = DecisionTreeClassifier(criterion='entropy',)
model.fit(X = df_to_model_train.loc[:, inference_cols],
          y = df_to_model_train['Response'])


DecisionTreeClassifier(criterion='entropy')

In [54]:
from sklearn.linear_model import LogisticRegression
logistic_model = LogisticRegression()
logistic_model.fit(X = df_to_model_train.loc[:, inference_cols],
                     y = df_to_model_train['Response']) 

/home/p123/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [31]:
y_pred = logistic_model.predict_proba(df_to_model_test.loc[:, inference_cols])
y_pred

array([[0.9969883 , 0.0030117 ],
       [0.86156091, 0.13843909],
       [0.99850084, 0.00149916],
       ...,
       [0.79947649, 0.20052351],
       [0.99772313, 0.00227687],
       [0.95013694, 0.04986306]])

In [48]:
def custom_predict(model,
                   data,
                   threshold=0.5):
    
    y_hat = model.predict_proba(data)

    return (y_hat[:, 1] > threshold).astype(int)

In [56]:
logistic_model.predict_proba(df_to_model_test.loc[:, inference_cols])

array([[0.9969883 , 0.0030117 ],
       [0.86156091, 0.13843909],
       [0.99850084, 0.00149916],
       ...,
       [0.79947649, 0.20052351],
       [0.99772313, 0.00227687],
       [0.95013694, 0.04986306]])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score , confusion_matrix

y_pred = custom_predict(model = logistic_model,
                        data = df_to_model_test.loc[:, inference_cols])

accuracy = accuracy_score(df_to_model_test['Response'],     y_pred=y_pred)
f1_score = f1_score(df_to_model_test['Response'],           y_pred=y_pred)
precision = precision_score(df_to_model_test['Response'],   y_pred=y_pred)
recall = recall_score(df_to_model_test['Response'],         y_pred=y_pred)

confusion_matrix = confusion_matrix(df_to_model_test['Response'], y_pred=y_pred)

print(f'Accuracy: {accuracy:.4f} | F1 Score: {f1_score:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}')

Accuracy: 0.8764 | F1 Score: 0.0079 | Precision: 0.2478 | Recall: 0.0040


In [46]:
confusion_matrix

array([[90223, 10097],
       [ 9791,  4222]])

In [27]:
df_to_model.Response.value_counts()

Response
0    334399
1     46710
Name: count, dtype: int64

In [28]:
importancies = model.feature_importances_
importancies = dict(zip(inference_cols , model.feature_importances_))
df_importancies = pd.DataFrame(importancies,
             index = [0]).T

df_importancies.columns = ['score']
df_importancies = df_importancies.reset_index(drop = False)
df_importancies = df_importancies.rename(columns={'0' : 'socre'})
df_importancies = df_importancies.sort_values(by = 'score',
                                              ascending=False)


df_importancies.reset_index()


,level_0,index,score
0,3,Previously_Insured,0.213793
1,0,id,0.186645
2,7,Vintage,0.160645
3,6,Annual_Premium,0.152246
4,1,Age,0.062409
5,5,Vehicle_Damage,0.026603
6,9,entre_30_50,0.017736
7,218,sexo_feminino,0.015447
8,193,region_code_28.0,0.009766
9,129,policy_sales_channel_124.0,0.007262


In [29]:
model.feature_importances_

array([1.86645065e-01, 6.24088933e-02, 4.61689104e-04, 2.13793109e-01,
       7.23414267e-03, 2.66025237e-02, 1.52246375e-01, 1.60644871e-01,
       3.61321038e-03, 1.77361030e-02, 2.36391463e-04, 2.83447718e-05,
       6.58256767e-04, 4.62063923e-04, 0.00000000e+00, 4.71099598e-04,
       6.36927923e-04, 2.49470237e-05, 2.18681035e-04, 4.76192373e-04,
       2.24295673e-04, 1.15894623e-03, 3.26260331e-04, 1.97697182e-04,
       2.47922262e-04, 0.00000000e+00, 0.00000000e+00, 1.50633890e-04,
       0.00000000e+00, 5.52713906e-05, 1.25998540e-04, 2.53275178e-04,
       5.04828228e-04, 1.26330251e-03, 4.06670988e-03, 0.00000000e+00,
       0.00000000e+00, 2.50368798e-04, 7.74011336e-04, 5.33153747e-04,
       3.03721024e-05, 0.00000000e+00, 0.00000000e+00, 3.99564814e-05,
       1.11998786e-04, 1.27588415e-04, 0.00000000e+00, 0.00000000e+00,
       3.38881746e-05, 0.00000000e+00, 2.90140307e-05, 0.00000000e+00,
       1.41384275e-04, 3.29874874e-05, 0.00000000e+00, 0.00000000e+00,
      

# 4 - Treinamento dos modelos

In [30]:
# trbalhar com as colunas abaixo
# Age, 
# Vehichle_Age
# Annual Premium 
# vintage, 
# Response